In [45]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.autograd import Variable

from torch.utils.data import Dataset,DataLoader

from tokenizers import ByteLevelBPETokenizer
from tokenizers.trainers import BpeTrainer
from tokenizers import Tokenizer, models, pre_tokenizers, decoders, trainers, processors
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace

from torchinfo import summary

САМ МИПЛ

In [46]:
class Miple(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, market_features, emotions_count, dropout):
        super(Miple, self).__init__()

        self.emb = nn.Embedding(vocab_size, embed_size)
        self.dropout = nn.Dropout(dropout)

        self.text_lstm = nn.LSTM(embed_size, hidden_size, batch_first=True)
        self.market_lstm = nn.LSTM(market_features, hidden_size, batch_first=True)
        combined_input_size = (hidden_size * 3) + emotions_count

        #обработка рынка
        self.decision = nn.Sequential(
            nn.Linear(combined_input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.Linear(hidden_size, market_features),
        )

        #обработка эмоций
        self.emotions_choicer = nn.Linear(combined_input_size, emotions_count)

        # общение
        self.combined_to_hidden = nn.Linear(combined_input_size, hidden_size)
        self.text_decoder_lstm = nn.LSTM(embed_size, hidden_size, batch_first=True)
        self.chat_message_writer = nn.Linear(hidden_size, vocab_size)

    def forward(self, market_seq, prompt_seq, news_seq, current_emotions,target_seq=None):
            # ембеддинги
            p_emb = self.emb(prompt_seq)
            n_emb = self.emb(news_seq)

            # взятие последнего состояния последнего слоя нейросети
            _, (h_p, _) = self.text_lstm(p_emb)
            _, (h_n, _) = self.text_lstm(n_emb)
            _, (h_m, _) = self.market_lstm(market_seq)

            combined = torch.cat((h_p[-1], h_n[-1], h_m[-1], current_emotions), dim=1)

            trade_logits = self.decision(combined)
            new_emotions = self.emotions_choicer(combined)

            decoder_hidden_init = torch.tanh(self.combined_to_hidden(combined)).unsqueeze(0) # [1, batch_size, hidden_size]
            decoder_cell_init = torch.zeros_hidden_like = torch.zeros_like(decoder_hidden_init)
            dec_states = (decoder_hidden_init, decoder_cell_init)

            if target_seq is not None:
                dec_emb = self.dropout(self.emb(target_seq))
                dec_outputs, _ = self.text_decoder_lstm(dec_emb, dec_states)
                text_logits = self.chat_message_writer(dec_outputs) # [batch_size, seq_len, vocab_size]
            else:
                text_logits = None

            return trade_logits, new_emotions, text_logits


Рассказчик (главный агент, который контролирует правила симуляции)

In [ ]:
class StoryTeller(nn.Module):
    def __init__(self, window_size: int, vocab_size: int, emb_size: int, rules_count: int):
        super(StoryTeller, self).__init__()

        self.time_window = nn.Sequential(
            nn.Linear(window_size, emb_size),
            nn.Linear(emb_size, rules_count)
        )

        self.chat_inflation = nn.Sequential(
                    nn.Linear(emb_size, rules_count)
                )
        self.embedding = nn.Embedding(vocab_size, emb_size)
        self.summary_layer = nn.Linear(rules_count * 2, rules_count)

    def forward(self, text, history):
        emb = self.embedding(text)
        history = self.time_window(history)
        emb_mean = torch.mean(emb, dim=1)
        chat_inf = self.chat_inflation(emb_mean)


        
        all_feautures = torch.cat((history, chat_inf), dim=1)
        res = self.summary_layer(all_feautures)

        return res


In [48]:
# Инициализируем модель с твоими параметрами
story_model = StoryTeller(window_size=32, vocab_size=4096, emb_size=256, rules_count=12)

# Создаем тестовые тензоры для проверки
batch_size = 2
seq_len = 128

dummy_text = torch.randint(0, 4096, (batch_size, seq_len), dtype=torch.long)
dummy_history = torch.randn(batch_size, 32, dtype=torch.float)

# Выводим красивую суммаризацию
model_stats = summary(
    story_model, 
    input_data=(dummy_text, dummy_history),
    col_names=["input_size", "output_size", "num_params", "kernel_size"],
    depth=3 # Глубина отображения вложенных слоев (nn.Sequential)
)

print(model_stats)


Layer (type:depth-idx)                   Input Shape               Output Shape              Param #                   Kernel Shape
StoryTeller                              [2, 128]                  [2, 12]                   --                        --
├─Embedding: 1-1                         [2, 128]                  [2, 128, 256]             1,048,576                 --
├─Sequential: 1-2                        [2, 32]                   [2, 12]                   --                        --
│    └─Linear: 2-1                       [2, 32]                   [2, 256]                  8,448                     --
│    └─Linear: 2-2                       [2, 256]                  [2, 12]                   3,084                     --
├─Sequential: 1-3                        [2, 256]                  [2, 12]                   --                        --
│    └─Linear: 2-3                       [2, 256]                  [2, 12]                   3,084                     --
├─Linear: 1-4 

In [49]:
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()
trainer = BpeTrainer(
    vocab_size=20000,
    special_tokens=["[PAD]", "[UNK]", "[SOS]", "[EOS]"],
    min_frequency=2
)

files = ["VocabText.txt"]
tokenizer.train(files, trainer)
tokenizer.save("tokenizer.json")


In [50]:
# тестовый прогон
vocab_size = tokenizer.get_vocab_size()
embed_size = 128
hidden_size = 256
market_features = 10
emotions_count = 4
dropout = 0.1

model = Miple(vocab_size, embed_size, hidden_size, market_features, emotions_count, dropout)

batch_size = 8
seq_len_text = 128    # длина текста
seq_len_market = 32  # длина истории рынка


market_input = torch.randn(batch_size, seq_len_market, market_features)
prompt_input = torch.randint(0, vocab_size, (batch_size, seq_len_text))
news_input = torch.randint(0, vocab_size, (batch_size, seq_len_text))
emotions_input = torch.randn(batch_size, emotions_count)


summary(model, input_data=[market_input, prompt_input, news_input, emotions_input])

Layer (type:depth-idx)                   Output Shape              Param #
Miple                                    [8, 10]                   1,143,134
├─Embedding: 1-1                         [8, 128, 128]             372,480
├─Embedding: 1-2                         [8, 128, 128]             (recursive)
├─LSTM: 1-3                              [8, 128, 256]             395,264
├─LSTM: 1-4                              [8, 128, 256]             (recursive)
├─LSTM: 1-5                              [8, 32, 256]              274,432
├─Sequential: 1-6                        [8, 10]                   --
│    └─Linear: 2-1                       [8, 256]                  197,888
│    └─ReLU: 2-2                         [8, 256]                  --
│    └─Linear: 2-3                       [8, 256]                  65,792
│    └─Linear: 2-4                       [8, 10]                   2,570
├─Linear: 1-7                            [8, 4]                    3,092
├─Linear: 1-8                 